# Exercise 8 — Better Pop a Pill?


## Task

A biochemical company offers nutrition pills for the alligator diet from Exercise 7. Each pill supplies **one marginal unit of exactly one nutrient**: protein, carbohydrate, or fat.

Determine the **maximum price per pill** that Max would be willing to pay for each nutrient. Then determine the corresponding willingness to pay for a combined pill that supplies one unit of **all three nutrients in a 1:1:1 ratio**.

Tasks:

1. Formulate the problem algebraically.
2. Solve it in JuMP using the alligator-diet model.
3. Interpret the result economically.
4. Determine the value of the combined 1:1:1 pill.

**Hint:** use LP duality. The shadow price of a binding nutrient requirement measures the marginal change in minimum feed cost when that requirement is increased by one unit.


## Reference implementation


In [1]:
using JuMP, HiGHS

model = Model(HiGHS.Optimizer)

nutrients = [:Protein, :Carbs, :Fat]
source = [:Chicken, :Kudu]

nut_data = [[2, 2, 0] [1, 4, 4]]

f = JuMP.Containers.DenseAxisArray(nut_data, nutrients, source)
r = JuMP.Containers.DenseAxisArray([6, 12, 4], nutrients)
c = JuMP.Containers.DenseAxisArray([5, 7], source)

@variable(model, x[source] >= 0)

@objective(model, Min, sum(c[s] * x[s] for s in source))
@constraint(model, min_req[n in nutrients], sum(f[n, s] * x[s] for s in source) >= r[n])

optimize!(model)
objective_value(model)
value.(x)
dual.(min_req)


Running HiGHS 1.15.1 (git hash: 04024d701f): Copyright (c) 2026 under MIT licence terms
Includes third-party software components, see THIRD_PARTY_NOTICES.md for full details
Using BLAS: Apple Accelerate 
LP has 3 rows; 2 cols; 5 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 4e+00]
  Cost    [5e+00, 7e+00]
  Bound   [0e+00, 0e+00]
  RHS     [4e+00, 1e+01]
Presolving model
2 rows, 2 cols, 4 nonzeros 0s
2 rows, 2 cols, 4 nonzeros 0s
Presolve reductions: rows 2(-1); columns 2(-0); nonzeros 4(-1) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0     7.0000055544e+00 Pr: 2(13) 0.0s
          2     2.4000000000e+01 Pr: 0(0) 0.0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model status        : Optimal
Simplex   iterations: 2
Objective value     :  2.4000000000e+01
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, [:Protein, :Carbs, :Fat]
And data, a 3-element Vector{Float64}:
 1.0
 1.5
 0.0

## Interpretation prompt

Inspect `dual.(min_req)`. For each nutrient, the corresponding dual value is the marginal cost of tightening that minimum requirement by one unit. Use those marginal values to obtain the willingness to pay for individual pills and add the three marginal values for a 1:1:1 combined pill.
